# The Bootstrap

Wiki reference for [bootstrap resampling](https://ml-viz-ruby.vercel.app/wiki/bootstrap-resampling).

**The idea in one sentence.** The bootstrap estimates the sampling distribution of *any*
statistic by **resampling the data with replacement** — no formula needed — which gives standard
errors and confidence intervals for things like the median that have no clean analytic SE, as
long as the statistic is **smooth** (it fails for extremes like the maximum).

We implement the bootstrap and several CI methods from scratch, **validate the $1-1/e$
resampling fact and near-nominal CI coverage**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(7)

## 1 — The bootstrap distribution

Resample with replacement B times, recompute the statistic each time.

In [ ]:
def bootstrap(data, stat_fn, B=5000, seed=0):
    """Return B bootstrap replicates of stat_fn applied to resamples of data."""
    rng_local = np.random.default_rng(seed)
    n = len(data)
    reps = np.empty(B)
    for b in range(B):
        resample = data[rng_local.integers(0, n, n)]   # sample with replacement
        reps[b] = stat_fn(resample)
    return reps

# Skewed data: the median has no clean analytic SE
data = rng.lognormal(mean=1.0, sigma=0.6, size=200)
reps_med = bootstrap(data, np.median, B=5000)

print(f"Sample median       = {np.median(data):.4f}")
print(f"Bootstrap SE(median)= {reps_med.std(ddof=1):.4f}")

plt.figure(figsize=(7, 4))
plt.hist(reps_med, bins=40, color='#6366f1', alpha=0.85)
plt.axvline(np.median(data), color='#f59e0b', label='sample median')
plt.xlabel('bootstrap median'); plt.ylabel('count')
plt.title('Bootstrap distribution of the median'); plt.legend()
plt.tight_layout(); plt.show()

## 2 — The ~63.2% inclusion fact

Each resample contains about $1 - e^{-1} = 63.2\%$ of the distinct original rows.

In [ ]:
n = 1000
fracs = []
for _ in range(2000):
    idx = rng.integers(0, n, n)
    fracs.append(len(np.unique(idx)) / n)
print(f"Mean distinct fraction = {np.mean(fracs):.4f}")
print(f"Theory 1 - 1/e         = {1 - np.exp(-1):.4f}")

### Validate: each resample contains ~63.2% of the data

Sampling $n$ points with replacement from $n$ leaves each original point out with probability
$(1-1/n)^n \to 1/e$, so on average a bootstrap sample contains $1 - 1/e \approx 63.2\%$ distinct
originals (the rest are duplicates). We confirm the fraction.

In [ ]:
print(f'mean distinct fraction = {np.mean(fracs):.4f}  vs  1 - 1/e = {1 - np.exp(-1):.4f}')
assert abs(np.mean(fracs) - (1 - np.exp(-1))) < 0.01, 'a bootstrap sample holds ~63.2% (1-1/e) of the originals'
print('\n✅ resampling with replacement: ~63% distinct points, the rest duplicates')

## 3 — Percentile, basic, and BCa intervals

In [ ]:
from scipy import stats

def percentile_ci(reps, alpha=0.05):
    return np.percentile(reps, [100*alpha/2, 100*(1-alpha/2)])

def basic_ci(reps, theta_hat, alpha=0.05):
    lo, hi = np.percentile(reps, [100*alpha/2, 100*(1-alpha/2)])
    return np.array([2*theta_hat - hi, 2*theta_hat - lo])

def bca_ci(data, stat_fn, reps, theta_hat, alpha=0.05):
    # bias-correction z0
    z0 = stats.norm.ppf((reps < theta_hat).mean())
    # acceleration via jackknife
    n = len(data)
    jk = np.array([stat_fn(np.delete(data, i)) for i in range(n)])
    jk_mean = jk.mean()
    num = ((jk_mean - jk)**3).sum()
    den = 6.0 * (((jk_mean - jk)**2).sum())**1.5
    a = num / den if den != 0 else 0.0
    z = stats.norm.ppf([alpha/2, 1 - alpha/2])
    adj = z0 + (z0 + z) / (1 - a*(z0 + z))
    pcts = 100 * stats.norm.cdf(adj)
    return np.percentile(reps, pcts)

theta_hat = np.median(data)
print(f"Percentile CI : {percentile_ci(reps_med).round(4)}")
print(f"Basic CI      : {basic_ci(reps_med, theta_hat).round(4)}")
print(f"BCa CI        : {bca_ci(data, np.median, reps_med, theta_hat).round(4)}")

## 4 — Coverage check

A 95% CI should contain the *true* parameter ~95% of the time. We simulate from a known distribution and count.

In [ ]:
true_mean = 5.0
cover_pct = 0
trials = 300
for t in range(trials):
    d = rng.normal(true_mean, 2.0, 60)
    reps = bootstrap(d, np.mean, B=1000, seed=t)
    lo, hi = percentile_ci(reps)
    cover_pct += (lo <= true_mean <= hi)
print(f"Percentile CI empirical coverage: {cover_pct/trials:.3f}  (target 0.95)")

### Validate: percentile CIs achieve near-nominal coverage

A 95% confidence interval should contain the true parameter ~95% of the time across repeated
experiments. We run 300 experiments and confirm the bootstrap percentile CI for the mean covers
the truth close to 95%.

In [ ]:
coverage = cover_pct / trials
print(f'percentile-CI empirical coverage over {trials} experiments: {coverage:.3f}  (target 0.95)')
assert 0.90 < coverage < 0.99, 'bootstrap percentile CIs achieve near-nominal coverage for a smooth statistic'
print('\n✅ the bootstrap gives calibrated intervals for smooth statistics — no formula required')

## 5 — Where the bootstrap fails: the maximum

The sample max can never exceed the largest observed value, so the bootstrap of `max` is degenerate.

In [ ]:
d = rng.uniform(0, 10, 100)
reps_max = bootstrap(d, np.max, B=5000)
print(f"Sample max = {d.max():.4f}")
print(f"Bootstrap max distribution takes only {len(np.unique(reps_max))} distinct values")
print(f"...and never exceeds the sample max: {reps_max.max():.4f}")
print("True max of Uniform(0,10) is 10 — the bootstrap upper tail is systematically too low.")

plt.figure(figsize=(7, 3.5))
plt.hist(reps_max, bins=30, color='#f87171', alpha=0.85)
plt.axvline(10, color='#f59e0b', label='true max = 10')
plt.title('Degenerate bootstrap distribution of the maximum'); plt.legend()
plt.tight_layout(); plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **extreme statistics** | bootstrap max/min are degenerate (demo) — use EVT |
| **dependent data** | i.i.d. resampling breaks autocorrelation; use block bootstrap |
| **too few replicates** | noisy CIs; use thousands of resamples |
| **percentile vs BCa** | BCa corrects skew/bias; percentile is simplest |
| **small n** | the bootstrap can't invent information the sample lacks |

Demo: the bootstrap distribution of the maximum is degenerate.

In [ ]:
# The bootstrap's key failure mode: EXTREME statistics. The maximum depends entirely on the
# most extreme observation, and a resample can never contain a value larger than the sample max
# — so the bootstrap distribution of the max is DEGENERATE (few distinct values) and its upper
# tail is systematically too low. We confirm on the Uniform(0,10) max.
print(f'sample max = {d.max():.3f}')
print(f'bootstrap-max distribution: {len(np.unique(reps_max))} distinct values, never exceeds {reps_max.max():.3f}')
assert reps_max.max() <= d.max() + 1e-12, 'a bootstrap resample can never exceed the sample max'
assert len(np.unique(reps_max)) < 20, 'the bootstrap max is degenerate — too few distinct values'
print('\nThe bootstrap works for SMOOTH statistics; for extremes (max/min) it fails -> use extreme-value theory.')

## ✏️ Your turn

**Task A — Block bootstrap:** For autocorrelated data, plain resampling underestimates the SE. Implement a moving block bootstrap: pick random contiguous blocks of length `ell` and concatenate them to length n. Compare the SE of the mean from the plain vs block bootstrap on an AR(1) series.

**Task B — Bootstrap a correlation:** Bootstrap Pearson's r between two correlated columns and report a BCa 95% CI. Verify the CI stays within [-1, 1] (a place where the basic interval can fail).

In [ ]:
# AR(1) series with strong autocorrelation
phi = 0.8
n = 400
series = np.zeros(n)
for t in range(1, n):
    series[t] = phi * series[t-1] + rng.normal()

def block_bootstrap_mean_se(series, ell, B=2000):
    """SE of the mean via moving block bootstrap with block length ell."""
    n = len(series)
    n_blocks = int(np.ceil(n / ell))
    means = np.empty(B)
    # TODO(you): for each replicate, pick n_blocks random start indices in [0, n-ell],
    # concatenate those blocks, truncate to length n, and record the mean
    return ...

plain_se = bootstrap(series, np.mean, B=2000).std(ddof=1)
block_se = block_bootstrap_mean_se(series, ell=20)
if block_se is not None:
    print(f"Plain bootstrap SE(mean) = {plain_se:.4f}  (too small — ignores autocorrelation)")
    print(f"Block bootstrap SE(mean) = {block_se:.4f}  (larger, more honest)")

<details><summary>Solution — Task A</summary>

```python
def block_bootstrap_mean_se(series, ell, B=2000):
    n = len(series)
    n_blocks = int(np.ceil(n / ell))
    means = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n - ell + 1, n_blocks)
        sample = np.concatenate([series[s:s+ell] for s in starts])[:n]
        means[b] = sample.mean()
    return means.std(ddof=1)
```

The block SE is larger because contiguous blocks preserve the AR(1) autocorrelation that plain resampling destroys.
</details>

## Key takeaways

- **Resample with replacement** to approximate any statistic's sampling distribution — no
  formula needed.
- **The $1-1/e$ fact:** each resample holds ~63.2% distinct points (verified).
- **Calibrated intervals:** percentile CIs achieve near-nominal coverage for smooth statistics
  (verified).
- **It fails for extremes:** the bootstrap max is degenerate (demo) — smoothness is required.